# Isolation Forest Based Network Anomaly Detection


## Objective


To detect anomalous network traffic (DDoS attacks) using the Isolation Forest algorithm and compare its performance against the Z-Score baseline established in the previous notebook.


## Background


Isolation Forest is an unsupervised anomaly detection algorithm based on the principle that anomalies are few and different, making them easier to isolate. Unlike distance-based or density-based methods, Isolation Forest explicitly isolates anomalies by randomly selecting a feature and then randomly selecting a split value between the maximum and minimum values of the selected feature. Anomalous observations require fewer splits (shorter path lengths) to be isolated, which forms the basis of the anomaly score.

In [ ]:
#Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.metrics import precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#Load Preprocessed Data

df = pd.read_csv("../data/ddos_clean.csv")

print("Dataset shape:", df.shape)
print("\nLabel distribution:")
print(df[" Label"].value_counts())
df.head()

In [ ]:
#Create Binary Anomaly Labels (same as notebook 03)

df["Anomaly"] = (
    df[" Label"] != "BENIGN"
).astype(int)

print("Anomaly distribution:")
print(df["Anomaly"].value_counts())
print(f"\nAnomaly ratio: {df['Anomaly'].mean():.4f}")

## Observation:

The anomaly ratio is approximately 0.57 (57%), meaning DDoS traffic constitutes the majority class. This is important for configuring the Isolation Forest contamination parameter, which represents the expected proportion of anomalies in the dataset.

In [ ]:
#Prepare Features and Labels

X = df.drop(
    [" Label", "Anomaly"],
    axis=1
)

y_true = df["Anomaly"]

print("Feature matrix shape:", X.shape)
print("Number of features:", X.shape[1])

In [ ]:
#Feature Scaling

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled feature matrix shape:", X_scaled.shape)
print("\nMean of first 5 features (should be ~0):")
print(np.mean(X_scaled, axis=0)[:5])
print("\nStd of first 5 features (should be ~1):")
print(np.std(X_scaled, axis=0)[:5])

## Observation:

Feature scaling was applied using StandardScaler to ensure all features contribute equally during the Isolation Forest's random feature selection and splitting process. Although Isolation Forest is relatively robust to feature scaling, standardization can still improve performance when features have vastly different ranges.

---

# Isolation Forest Training

The key hyperparameters for Isolation Forest are:

- **n_estimators**: Number of isolation trees in the ensemble
- **contamination**: Expected proportion of anomalies in the dataset
- **max_samples**: Number of samples to draw for training each tree
- **random_state**: Seed for reproducibility

In [ ]:
#Train Isolation Forest with default contamination (auto)

iso_forest_auto = IsolationForest(
    n_estimators=100,
    contamination='auto',
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)

iso_forest_auto.fit(X_scaled)

# Predict: Isolation Forest returns -1 for anomalies and 1 for normal
y_pred_auto_raw = iso_forest_auto.predict(X_scaled)

# Convert to binary: -1 (anomaly) -> 1, 1 (normal) -> 0
y_pred_auto = (y_pred_auto_raw == -1).astype(int)

print("Contamination = 'auto'")
print("Prediction distribution:")
print(pd.Series(y_pred_auto).value_counts())

In [ ]:
#Evaluation for contamination='auto'

print("Classification Report (contamination='auto'):")
print(classification_report(y_true, y_pred_auto))

cm_auto = confusion_matrix(y_true, y_pred_auto)
print("Confusion Matrix:")
print(cm_auto)

## Observation:

With contamination set to 'auto' (which defaults to approximately 0.5 in scikit-learn), the Isolation Forest flags a relatively small number of samples as anomalous. Since the actual anomaly proportion is ~57%, the 'auto' setting underestimates the contamination level, leading to a lower recall for detecting DDoS traffic.

---

## Experimenting with Different Contamination Rates

Since the actual DDoS proportion is ~57%, we will evaluate multiple contamination rates to find the optimal configuration.

In [ ]:
#Define evaluation function for Isolation Forest

def evaluate_isolation_forest(contamination, X_scaled, y_true):
    """
    Train and evaluate an Isolation Forest model with the given contamination rate.
    Returns a dictionary of metrics.
    """
    iso_forest = IsolationForest(
        n_estimators=100,
        contamination=contamination,
        max_samples='auto',
        random_state=42,
        n_jobs=-1
    )

    iso_forest.fit(X_scaled)

    y_pred_raw = iso_forest.predict(X_scaled)
    y_pred = (y_pred_raw == -1).astype(int)

    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"\n{'='*50}")
    print(f"Contamination = {contamination}")
    print(f"{'='*50}")
    print(f"Predicted anomalies: {y_pred.sum()}")
    print(f"Actual anomalies:    {y_true.sum()}")
    print(f"\nPrecision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    return {
        'contamination': contamination,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'y_pred': y_pred,
        'model': iso_forest
    }

In [ ]:
#Evaluate multiple contamination rates

contamination_rates = [0.1, 0.2, 0.3, 0.4, 0.5]

results = []
for rate in contamination_rates:
    result = evaluate_isolation_forest(rate, X_scaled, y_true)
    results.append(result)

In [ ]:
#Summary Table

results_df = pd.DataFrame([
    {
        'Contamination': r['contamination'],
        'Precision': round(r['precision'], 4),
        'Recall': round(r['recall'], 4),
        'F1-Score': round(r['f1'], 4)
    }
    for r in results
])

print("\nIsolation Forest - Contamination Rate Comparison:")
print(results_df.to_string(index=False))

## Observation:

The Isolation Forest was evaluated across five contamination rates (0.1 to 0.5). As the contamination rate increases, recall improves since more samples are flagged as anomalous. However, precision may decrease due to more false positives. The optimal contamination rate balances precision and recall, as reflected in the F1-score.

---

## Visualization of Results

In [ ]:
#Bar Chart: Contamination Rate Comparison

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(results_df))
width = 0.25

bars1 = ax.bar(x - width, results_df['Precision'], width, label='Precision', color='#2196F3')
bars2 = ax.bar(x, results_df['Recall'], width, label='Recall', color='#FF9800')
bars3 = ax.bar(x + width, results_df['F1-Score'], width, label='F1-Score', color='#4CAF50')

ax.set_xlabel('Contamination Rate')
ax.set_ylabel('Score')
ax.set_title('Isolation Forest - Contamination Rate Comparison')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Contamination'])
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(
    "../figures/iforest_contamination_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
#Line Plot: Metrics vs Contamination Rate

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(results_df['Contamination'], results_df['Precision'],
        marker='o', linewidth=2, label='Precision', color='#2196F3')
ax.plot(results_df['Contamination'], results_df['Recall'],
        marker='s', linewidth=2, label='Recall', color='#FF9800')
ax.plot(results_df['Contamination'], results_df['F1-Score'],
        marker='^', linewidth=2, label='F1-Score', color='#4CAF50')

ax.set_xlabel('Contamination Rate')
ax.set_ylabel('Score')
ax.set_title('Isolation Forest - Metrics vs Contamination Rate')
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(
    "../figures/iforest_metrics_vs_contamination.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## Observation:

The line plot reveals the trade-off between precision and recall as the contamination rate increases. The F1-score curve helps identify the optimal contamination rate that balances both metrics.

---

## Best Model Selection and Confusion Matrix

In [ ]:
#Select best model based on F1-Score

best_result = max(results, key=lambda r: r['f1'])

print(f"Best contamination rate: {best_result['contamination']}")
print(f"Best F1-Score: {best_result['f1']:.4f}")
print(f"Precision: {best_result['precision']:.4f}")
print(f"Recall: {best_result['recall']:.4f}")

In [ ]:
#Confusion Matrix for Best Model

fig, ax = plt.subplots(figsize=(8, 6))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    best_result['y_pred'],
    ax=ax
)

ax.set_title(
    f"Isolation Forest Confusion Matrix\n(contamination={best_result['contamination']})"
)

plt.savefig(
    "../figures/iforest_best_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
#Confusion Matrices for All Contamination Rates

fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 5))

for idx, result in enumerate(results):
    ConfusionMatrixDisplay.from_predictions(
        y_true,
        result['y_pred'],
        ax=axes[idx]
    )
    axes[idx].set_title(
        f"contamination={result['contamination']}\n"
        f"F1={result['f1']:.3f}"
    )

plt.suptitle("Isolation Forest - Confusion Matrices Across Contamination Rates", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(
    "../figures/iforest_all_confusion_matrices.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## Observation:

The confusion matrices across different contamination rates illustrate how increasing the contamination parameter shifts the decision boundary. At lower contamination rates, fewer samples are flagged as anomalies, resulting in higher precision but lower recall. At higher contamination rates, more true DDoS flows are detected, but at the cost of increased false positives.

---

## Anomaly Score Analysis

In [ ]:
#Anomaly Score Distribution

best_model = best_result['model']
anomaly_scores = best_model.decision_function(X_scaled)

print("Anomaly Score Statistics:")
print(f"  Min:    {anomaly_scores.min():.4f}")
print(f"  Max:    {anomaly_scores.max():.4f}")
print(f"  Mean:   {anomaly_scores.mean():.4f}")
print(f"  Median: {np.median(anomaly_scores):.4f}")
print(f"  Std:    {anomaly_scores.std():.4f}")

In [ ]:
#Anomaly Score Distribution by Class

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(
    anomaly_scores[y_true == 0],
    bins=50, alpha=0.6, label='BENIGN', color='#2196F3'
)
axes[0].hist(
    anomaly_scores[y_true == 1],
    bins=50, alpha=0.6, label='DDoS', color='#F44336'
)
axes[0].set_xlabel('Anomaly Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Anomaly Score Distribution by Class')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Boxplot
score_df = pd.DataFrame({
    'Anomaly Score': anomaly_scores,
    'Class': ['BENIGN' if y == 0 else 'DDoS' for y in y_true]
})

score_df.boxplot(
    column='Anomaly Score',
    by='Class',
    ax=axes[1]
)
axes[1].set_title('Anomaly Score Boxplot by Class')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Anomaly Score')
plt.suptitle('')  # Remove auto-generated title

plt.tight_layout()
plt.savefig(
    "../figures/iforest_anomaly_score_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## Observation:

The anomaly score distribution reveals how well the Isolation Forest separates BENIGN and DDoS traffic. Lower anomaly scores (more negative) indicate more anomalous observations. If there is clear separation between the two distributions, the Isolation Forest is effectively learning the difference between normal and attack traffic. Significant overlap indicates that some DDoS flows resemble normal traffic, making detection challenging.

---

## Feature Importance Analysis

In [ ]:
#Feature Importance using Mean Path Length
#
# Isolation Forest does not have a built-in feature_importances_ attribute.
# We approximate feature importance by measuring how much each feature
# contributes to the anomaly score using a permutation-based approach.

from sklearn.inspection import permutation_importance

# Use anomaly score as the target for permutation importance
# We evaluate which features, when shuffled, most change the anomaly scores

best_model_for_importance = best_result['model']

# Sample for computational efficiency
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), size=min(10000, len(X_scaled)), replace=False)
X_sample = X_scaled[sample_idx]
y_sample = y_true.values[sample_idx]

perm_importance = permutation_importance(
    best_model_for_importance,
    X_sample,
    y_sample,
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
    scoring='f1'
)

feature_names = X.columns.tolist()

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance Mean': perm_importance.importances_mean,
    'Importance Std': perm_importance.importances_std
}).sort_values('Importance Mean', ascending=False)

print("Top 15 Most Important Features:")
print(importance_df.head(15).to_string(index=False))

In [ ]:
#Feature Importance Visualization (Top 15)

top_n = 15
top_features = importance_df.head(top_n)

fig, ax = plt.subplots(figsize=(10, 8))

ax.barh(
    range(top_n),
    top_features['Importance Mean'].values,
    xerr=top_features['Importance Std'].values,
    color='#2196F3',
    alpha=0.8,
    edgecolor='black',
    linewidth=0.5
)

ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features['Feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Permutation Importance (Mean Decrease in F1)')
ax.set_title('Isolation Forest - Top 15 Feature Importances')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(
    "../figures/iforest_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## Observation:

The permutation importance analysis reveals which network flow features are most critical for the Isolation Forest's ability to distinguish DDoS traffic from normal traffic. Features with high importance scores are the ones that, when shuffled, cause the largest drop in detection performance. These features represent the most discriminative characteristics of DDoS attack patterns.

---

## Hyperparameter Tuning: Number of Estimators

In [ ]:
#Effect of n_estimators on performance

best_contamination = best_result['contamination']

n_estimators_list = [50, 100, 200, 300]
estimator_results = []

for n_est in n_estimators_list:
    iso_forest = IsolationForest(
        n_estimators=n_est,
        contamination=best_contamination,
        max_samples='auto',
        random_state=42,
        n_jobs=-1
    )

    iso_forest.fit(X_scaled)

    y_pred_raw = iso_forest.predict(X_scaled)
    y_pred = (y_pred_raw == -1).astype(int)

    f1 = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)

    estimator_results.append({
        'n_estimators': n_est,
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4)
    })

    print(f"n_estimators={n_est}: Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}")

est_df = pd.DataFrame(estimator_results)
print("\nn_estimators Comparison:")
print(est_df.to_string(index=False))

In [ ]:
#Visualize n_estimators effect

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(est_df['n_estimators'], est_df['Precision'],
        marker='o', linewidth=2, label='Precision', color='#2196F3')
ax.plot(est_df['n_estimators'], est_df['Recall'],
        marker='s', linewidth=2, label='Recall', color='#FF9800')
ax.plot(est_df['n_estimators'], est_df['F1-Score'],
        marker='^', linewidth=2, label='F1-Score', color='#4CAF50')

ax.set_xlabel('Number of Estimators')
ax.set_ylabel('Score')
ax.set_title(f'Isolation Forest - Effect of n_estimators (contamination={best_contamination})')
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(
    "../figures/iforest_n_estimators_effect.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## Observation:

The number of estimators (trees) has a moderate impact on performance. Increasing the number of trees generally leads to more stable predictions, though the improvement diminishes beyond a certain point. This is consistent with ensemble learning theory, where additional weak learners yield diminishing marginal returns.

---

## Comparison with Z-Score Baseline

In [ ]:
#Comparison Table: Isolation Forest vs Z-Score Baseline

# Z-Score results from notebook 03
zscore_results = pd.DataFrame({
    'Method': ['Z-Score (t=3.0)', 'Z-Score (t=2.5)', 'Z-Score (t=2.0)'],
    'Precision': [0.34, 0.27, 0.30],
    'Recall': [0.20, 0.22, 0.28],
    'F1-Score': [0.25, 0.24, 0.29]
})

# Best Isolation Forest results
iforest_best = pd.DataFrame({
    'Method': [f'Isolation Forest (c={best_result["contamination"]})'],
    'Precision': [round(best_result['precision'], 2)],
    'Recall': [round(best_result['recall'], 2)],
    'F1-Score': [round(best_result['f1'], 2)]
})

# Add all IF results
iforest_all = pd.DataFrame([
    {
        'Method': f'Isolation Forest (c={r["contamination"]})',
        'Precision': round(r['precision'], 2),
        'Recall': round(r['recall'], 2),
        'F1-Score': round(r['f1'], 2)
    }
    for r in results
])

comparison_df = pd.concat([zscore_results, iforest_all], ignore_index=True)

print("\n" + "="*65)
print("    COMPARISON: Z-Score vs Isolation Forest")
print("="*65)
print(comparison_df.to_string(index=False))
print("="*65)

In [ ]:
#Visual Comparison: Z-Score vs Isolation Forest

fig, ax = plt.subplots(figsize=(12, 6))

methods = comparison_df['Method'].tolist()
x = np.arange(len(methods))
width = 0.25

bars1 = ax.bar(x - width, comparison_df['Precision'], width,
               label='Precision', color='#2196F3')
bars2 = ax.bar(x, comparison_df['Recall'], width,
               label='Recall', color='#FF9800')
bars3 = ax.bar(x + width, comparison_df['F1-Score'], width,
               label='F1-Score', color='#4CAF50')

ax.set_xlabel('Method')
ax.set_ylabel('Score')
ax.set_title('Z-Score vs Isolation Forest - Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(
            f'{height:.2f}',
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),
            textcoords='offset points',
            ha='center', va='bottom',
            fontsize=8
        )

plt.tight_layout()
plt.savefig(
    "../figures/zscore_vs_iforest_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## Observation:

The visual comparison clearly shows the performance difference between the Z-Score baseline and the Isolation Forest approach. The Isolation Forest, being a machine learning-based method, is expected to capture more complex, non-linear patterns in the network traffic data compared to the simple statistical threshold-based Z-Score method.

---

## Summary and Conclusion

### Key Findings

1. **Isolation Forest** was applied to the CICIDS2017 DDoS dataset as an unsupervised anomaly detection method.

2. **Multiple contamination rates** (0.1 to 0.5) were evaluated to find the optimal configuration.

3. **Hyperparameter tuning** of n_estimators showed that increasing the number of trees leads to more stable but marginally improving results.

4. **Feature importance analysis** identified the most discriminative network flow features for DDoS detection.

5. **Compared to the Z-Score baseline** (best F1 = 0.29), the Isolation Forest demonstrates the ability of ML-based methods to capture more complex anomaly patterns in network traffic data.


### Limitations

1. Isolation Forest is an unsupervised method and does not leverage the available ground-truth labels during training.

2. The contamination parameter requires prior knowledge about the expected anomaly proportion.

3. For datasets where anomalies are not well-isolated in the feature space, Isolation Forest may still struggle.


### Next Steps

1. Evaluate supervised classification methods (e.g., Random Forest, XGBoost) that can leverage labeled training data.

2. Investigate deep learning-based approaches (e.g., Autoencoders) for anomaly detection.

3. Extend the analysis to other attack types in the CICIDS2017 dataset.